
<div align="center">

<h1>CAPSTONE PROJECT 3</h1>

<h2>Social Media Usage and Emotional Well-Being</h2>

<h3>Classification Using Machine Learning and Deep Learning</h3>

<hr>

<h3>Prepared By</h3>

<h2>Bhushan Mandekar</h2>

<p><b>Development Environment:</b> Visual Studio Code</p>
<p><b>Programming Language:</b> Python</p>
<p><b>Dataset Source:</b> Kaggle</p>

<hr>

<h3>PROJECT OBJECTIVE</h3>

<p>
To develop and evaluate machine learning and deep learning
classification models for predicting dominant emotional
states based on social media usage and engagement patterns.
</p>

</div>


Import all required libraries

In [ ]:

import os
import csv
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


: 

Configure project paths

In [ ]:

BASE_DIR = Path.cwd()

if not (BASE_DIR / "train.csv").exists():
    raise FileNotFoundError(
        "train.csv not found. Open the correct "
        "VS Code project folder."
    )

DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
REPORT_DIR = BASE_DIR / "reports"

for folder in [
    DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
    FIGURE_DIR,
    REPORT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = BASE_DIR / "train.csv"
VAL_PATH = BASE_DIR / "val.csv"
TEST_PATH = BASE_DIR / "test.csv"

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert path.exists(), f"Missing file: {path}"

print("Project folder:", BASE_DIR)
print("Training CSV:", TRAIN_PATH)
print("Validation CSV:", VAL_PATH)
print("Testing CSV:", TEST_PATH)


Inspect raw CSV structure

In [ ]:

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print("\n" + "=" * 60)
    print("File:", path.name)

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:
        rows = list(csv.reader(f))

    print("Header:", rows[0])
    print("Total CSV lines:", len(rows))

    widths = pd.Series(
        [len(row) for row in rows]
    ).value_counts().sort_index()

    print("Fields per row:")
    print(widths)


Load datasets with a cleaning audit

In [ ]:

EXPECTED_COLUMNS = [
    "User_ID",
    "Age",
    "Gender",
    "Platform",
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day",
    "Dominant_Emotion"
]

parsing_issues = []

def load_csv_checked(file_path):
    valid_rows = []

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.reader(f)
        header = next(reader)

        if header != EXPECTED_COLUMNS:
            raise ValueError(
                f"Unexpected columns in {file_path.name}: "
                f"{header}"
            )

        for line_no, row in enumerate(reader, start=2):

            if not row or all(
                not str(value).strip() for value in row
            ):
                continue

            if len(row) == len(EXPECTED_COLUMNS):
                valid_rows.append(row)

            else:
                parsing_issues.append({
                    "File": file_path.name,
                    "Line": line_no,
                    "Field_Count": len(row),
                    "Raw_Row": repr(row),
                    "Action": "Excluded pending review"
                })

    return pd.DataFrame(
        valid_rows,
        columns=EXPECTED_COLUMNS
    )

train_raw = load_csv_checked(TRAIN_PATH)
val_raw = load_csv_checked(VAL_PATH)
test_raw = load_csv_checked(TEST_PATH)

parsing_audit = pd.DataFrame(
    parsing_issues,
    columns=[
        "File", "Line", "Field_Count",
        "Raw_Row", "Action"
    ]
)

parsing_audit.to_csv(
    OUTPUT_DIR / "parsing_audit.csv",
    index=False
)

print("Training:", train_raw.shape)
print("Validation:", val_raw.shape)
print("Testing:", test_raw.shape)

print("\nMalformed rows requiring review:")
display(parsing_audit)


## Data Cleaning and Preprocessing

This section performs data cleaning and quality validation
on the training, validation, and testing datasets.

The objectives are to:
- Correct data entry errors and swapped column values.
- Standardize categorical variables.
- Validate numeric and categorical features.
- Identify missing values and duplicate records.
- Prevent data leakage between datasets.
- Save cleaned datasets and data quality reports.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data cleaning libraries imported successfully!")

Create working copies of the original datasets

In [ ]:
# Always begin cleaning from original loaded datasets

train_df = train_raw.copy()
val_df = val_raw.copy()
test_df = test_raw.copy()

print("Training Dataset:", train_df.shape)
print("Validation Dataset:", val_df.shape)
print("Testing Dataset:", test_df.shape)

Define dataset columns and valid categories

In [ ]:
# =====================================================
# DEFINE DATASET SCHEMA
# =====================================================

ID_COLUMN = "User_ID"

TARGET_COLUMN = "Dominant_Emotion"

NUMERIC_COLUMNS = [
    "Age",
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day"
]

COUNT_COLUMNS = [
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day"
]

CATEGORICAL_COLUMNS = [
    "Gender",
    "Platform"
]

VALID_GENDERS = [
    "Male",
    "Female",
    "Non-binary"
]

VALID_EMOTIONS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Anxiety",
    "Boredom",
    "Neutral"
]

EXPECTED_COLUMNS = (
    [ID_COLUMN]
    + NUMERIC_COLUMNS
    + CATEGORICAL_COLUMNS
    + [TARGET_COLUMN]
)

print("Dataset schema defined successfully!")

Verify dataset structure

In [ ]:
# =====================================================
# VERIFY REQUIRED COLUMNS
# =====================================================

datasets = {
    "Training": train_df,
    "Validation": val_df,
    "Testing": test_df
}

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("Shape:", df.shape)

    missing_columns = [
        col for col in EXPECTED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name} missing columns: {missing_columns}"
        )

    print("All required columns present.")

    display(df.head())

Examine missing values before cleaning

In [ ]:
# =====================================================
# INITIAL MISSING VALUE ANALYSIS
# =====================================================

missing_summary = []

for name, df in datasets.items():

    for col in df.columns:

        missing_summary.append({
            "Dataset": name,
            "Column": col,
            "Missing_Count": int(df[col].isna().sum()),
            "Missing_Percentage": round(
                df[col].isna().mean() * 100, 2
            )
        })

missing_summary = pd.DataFrame(missing_summary)

display(missing_summary)

missing_summary.to_csv(
    OUTPUT_DIR / "initial_missing_values.csv",
    index=False
)

Standardize text formatting

In [ ]:
# =====================================================
# REMOVE EXTRA SPACES AND STANDARDIZE TEXT
# =====================================================

def standardize_text(df):

    df = df.copy()

    text_columns = [
        "Age",
        "Gender",
        "Platform",
        TARGET_COLUMN
    ]

    for col in text_columns:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
        )

        df[col] = df[col].replace(
            ["", "nan", "None", "null", "N/A"],
            pd.NA
        )

    return df


train_df = standardize_text(train_df)
val_df = standardize_text(val_df)
test_df = standardize_text(test_df)

print("Text standardization completed!")

Correct swapped Age and Gender values

In [ ]:
# =====================================================
# CORRECT SWAPPED AGE AND GENDER VALUES
# =====================================================

GENDER_MAPPING = {
    "male": "Male",
    "female": "Female",
    "non-binary": "Non-binary",
    "non binary": "Non-binary",
    "nonbinary": "Non-binary"
}

def fix_age_gender(df, name):

    df = df.copy()

    age_text = df["Age"].astype("string").str.lower()

    gender_numeric = pd.to_numeric(
        df["Gender"],
        errors="coerce"
    )

    swap_mask = (
        age_text.isin(GENDER_MAPPING.keys())
        &
        gender_numeric.between(13, 100)
    ).fillna(False)

    print(f"{name} swapped rows:", swap_mask.sum())

    old_age = df.loc[swap_mask, "Age"].copy()
    old_gender = df.loc[swap_mask, "Gender"].copy()

    df.loc[swap_mask, "Age"] = old_gender.to_numpy()
    df.loc[swap_mask, "Gender"] = old_age.to_numpy()

    df["Gender"] = (
        df["Gender"]
        .astype("string")
        .str.lower()
        .map(GENDER_MAPPING)
        .fillna(df["Gender"])
    )

    return df


train_df = fix_age_gender(train_df, "Training")
val_df = fix_age_gender(val_df, "Validation")
test_df = fix_age_gender(test_df, "Testing")

Standardize platform and emotion categories

In [ ]:
# =====================================================
# STANDARDIZE CATEGORICAL VALUES
# =====================================================

PLATFORM_MAPPING = {
    "instagram": "Instagram",
    "facebook": "Facebook",
    "twitter": "Twitter",
    "linkedin": "LinkedIn",
    "snapchat": "Snapchat",
    "whatsapp": "WhatsApp",
    "telegram": "Telegram",
    "tiktok": "TikTok",
    "youtube": "YouTube"
}

EMOTION_MAPPING = {
    emotion.lower(): emotion
    for emotion in VALID_EMOTIONS
}

def standardize_categories(df):

    df = df.copy()

    df["Platform"] = (
        df["Platform"]
        .astype("string")
        .str.lower()
        .map(PLATFORM_MAPPING)
        .fillna(df["Platform"])
    )

    df[TARGET_COLUMN] = (
        df[TARGET_COLUMN]
        .astype("string")
        .str.lower()
        .map(EMOTION_MAPPING)
        .fillna(df[TARGET_COLUMN])
    )

    return df


train_df = standardize_categories(train_df)
val_df = standardize_categories(val_df)
test_df = standardize_categories(test_df)

print("Categorical standardization completed!")

Convert numeric columns

In [ ]:
# =====================================================
# NUMERIC DATA TYPE CONVERSION
# =====================================================

def convert_numeric(df):

    df = df.copy()

    for col in NUMERIC_COLUMNS:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df[ID_COLUMN] = pd.to_numeric(
        df[ID_COLUMN],
        errors="coerce"
    )

    return df


train_df = convert_numeric(train_df)
val_df = convert_numeric(val_df)
test_df = convert_numeric(test_df)

print("Numeric conversion completed!")

print("\nTraining Data Types:")
print(train_df.dtypes)

Identify invalid records

In [ ]:
# =====================================================
# DATA QUALITY VALIDATION
# =====================================================

def identify_invalid_records(df):

    flags = pd.DataFrame(index=df.index)

    flags["Invalid_Age"] = (
        df["Age"].isna()
        |
        ~df["Age"].between(13, 100)
    ).fillna(True)

    flags["Invalid_Gender"] = (
        ~df["Gender"].isin(VALID_GENDERS)
    ).fillna(True)

    flags["Invalid_Emotion"] = (
        ~df[TARGET_COLUMN].isin(VALID_EMOTIONS)
    ).fillna(True)

    flags["Missing_Platform"] = (
        df["Platform"].isna()
    )

    flags["Invalid_User_ID"] = (
        df[ID_COLUMN].isna()
        |
        (df[ID_COLUMN] % 1 != 0)
    ).fillna(True)

    flags["Missing_Numeric"] = (
        df[NUMERIC_COLUMNS].isna().any(axis=1)
    )

    flags["Negative_Numeric"] = (
        (df[NUMERIC_COLUMNS] < 0).any(axis=1)
    )

    flags["Invalid_Counts"] = (
        (
            df[COUNT_COLUMNS].notna()
            &
            (df[COUNT_COLUMNS] % 1 != 0)
        ).any(axis=1)
    )

    flags["Invalid_Usage_Time"] = (
        ~df["Daily_Usage_Time (minutes)"]
        .between(0, 1440)
    ).fillna(True)

    return flags.fillna(True)


train_flags = identify_invalid_records(train_df)
val_flags = identify_invalid_records(val_df)
test_flags = identify_invalid_records(test_df)

print("Training invalid flags:")
print(train_flags.sum())

print("\nValidation invalid flags:")
print(val_flags.sum())

print("\nTesting invalid flags:")
print(test_flags.sum())

Separate invalid records and create cleaned datasets

In [ ]:
# =====================================================
# SEPARATE INVALID RECORDS
# =====================================================

def separate_invalid(df, flags, name):

    invalid_mask = flags.any(axis=1)

    excluded = df.loc[invalid_mask].copy()

    for col in flags.columns:
        excluded[col] = flags.loc[
            invalid_mask, col
        ]

    cleaned = df.loc[
        ~invalid_mask
    ].copy()

    print(f"\n{name}")
    print("Original:", len(df))
    print("Excluded:", len(excluded))
    print("Remaining:", len(cleaned))

    excluded.to_csv(
        OUTPUT_DIR / f"{name.lower()}_invalid_records.csv",
        index=False
    )

    return cleaned, excluded


train_df, excluded_train = separate_invalid(
    train_df, train_flags, "Training"
)

val_df, excluded_val = separate_invalid(
    val_df, val_flags, "Validation"
)

test_df, excluded_test = separate_invalid(
    test_df, test_flags, "Testing"
)

Detect exact duplicates within each dataset

In [ ]:
# =====================================================
# EXACT DUPLICATE DETECTION
# =====================================================

for name, df in {
    "Training": train_df,
    "Validation": val_df,
    "Testing": test_df
}.items():

    duplicate_mask = df.duplicated(keep="first")

    print(
        name,
        "exact duplicate rows:",
        duplicate_mask.sum()
    )

    duplicates = df.loc[duplicate_mask]

    duplicates.to_csv(
        OUTPUT_DIR / f"{name.lower()}_duplicates.csv",
        index=False
    )

Remove exact duplicate rows

In [ ]:
# =====================================================
# REMOVE EXACT DUPLICATES WITHIN EACH DATASET
# =====================================================

train_df = (
    train_df
    .drop_duplicates()
    .reset_index(drop=True)
)

val_df = (
    val_df
    .drop_duplicates()
    .reset_index(drop=True)
)

test_df = (
    test_df
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Check overlap between datasets

In [ ]:
# =====================================================
# TRAIN-VALIDATION-TEST OVERLAP DETECTION
# =====================================================

RECORD_COLUMNS = [
    col for col in EXPECTED_COLUMNS
    if col != ID_COLUMN
]

def record_keys(df):

    return set(
        map(
            tuple,
            df[RECORD_COLUMNS]
            .fillna("__MISSING__")
            .astype(str)
            .to_numpy()
        )
    )


train_keys = record_keys(train_df)
val_keys = record_keys(val_df)
test_keys = record_keys(test_df)

print(
    "Train-Val overlapping records:",
    len(train_keys & val_keys)
)

print(
    "Train-Test overlapping records:",
    len(train_keys & test_keys)
)

print(
    "Val-Test overlapping records:",
    len(val_keys & test_keys)
)

print("\nUser_ID overlap:")

print(
    "Train-Val:",
    len(
        set(train_df[ID_COLUMN])
        &
        set(val_df[ID_COLUMN])
    )
)

print(
    "Train-Test:",
    len(
        set(train_df[ID_COLUMN])
        &
        set(test_df[ID_COLUMN])
    )
)

print(
    "Val-Test:",
    len(
        set(val_df[ID_COLUMN])
        &
        set(test_df[ID_COLUMN])
    )
)

In [ ]:
# =====================================================
# INSPECT CROSS-DATASET OVERLAP
# =====================================================

RECORD_COLUMNS = [
    col for col in EXPECTED_COLUMNS
    if col != ID_COLUMN
]

def get_record_keys(df):
    return pd.Series(
        list(
            map(
                tuple,
                df[RECORD_COLUMNS]
                .fillna("__MISSING__")
                .astype(str)
                .to_numpy()
            )
        ),
        index=df.index
    )

train_keys = set(get_record_keys(train_df))

val_record_keys = get_record_keys(val_df)

val_overlap_mask = val_record_keys.isin(train_keys)

val_after_overlap = val_df.loc[
    ~val_overlap_mask
].copy()

reference_keys = (
    train_keys
    |
    set(get_record_keys(val_after_overlap))
)

test_record_keys = get_record_keys(test_df)

test_overlap_mask = test_record_keys.isin(
    reference_keys
)

test_after_overlap = test_df.loc[
    ~test_overlap_mask
].copy()

print("=" * 60)
print("CROSS-DATASET OVERLAP IMPACT")
print("=" * 60)

print("Training records:", len(train_df))

print(
    "Validation records before:",
    len(val_df),
    "| After:",
    len(val_after_overlap)
)

print(
    "Testing records before:",
    len(test_df),
    "| After:",
    len(test_after_overlap)
)

Inspect duplicate and conflicting observations

In [ ]:
# =====================================================
# INSPECT UNIQUE AND CONFLICTING RECORDS
# =====================================================

# Combine cleaned datasets for investigation only
combined_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

print("Combined records:", len(combined_df))

# Exact duplicate records, excluding User_ID
record_columns = [
    col for col in combined_df.columns
    if col != "User_ID"
]

unique_df = combined_df.drop_duplicates(
    subset=record_columns
).copy()

print("Unique feature + target records:", len(unique_df))

# Check whether identical features have different labels
feature_columns = [
    col for col in record_columns
    if col != "Dominant_Emotion"
]

label_counts = (
    combined_df
    .groupby(feature_columns, dropna=False)
    ["Dominant_Emotion"]
    .nunique()
)

conflicting_groups = label_counts[
    label_counts > 1
]

print(
    "Feature combinations with conflicting emotions:",
    len(conflicting_groups)
)

# Check whether User_ID values are repeated
print(
    "Unique User_ID values:",
    combined_df["User_ID"].nunique()
)

print(
    "Repeated User_ID rows:",
    combined_df["User_ID"].duplicated().sum()
)

print("\nEmotion distribution in unique records:")
print(unique_df["Dominant_Emotion"].value_counts())

Check User_ID consistency

In [ ]:
# =====================================================
# CHECK USER_ID CONSISTENCY
# =====================================================

# Number of distinct feature/target records per User_ID
user_variations = (
    combined_df
    .groupby("User_ID")[record_columns]
    .nunique(dropna=False)
)

# IDs associated with different values in any column
inconsistent_users = user_variations[
    user_variations.gt(1).any(axis=1)
]

print("=" * 60)
print("USER_ID CONSISTENCY CHECK")
print("=" * 60)

print("Total unique User_IDs:",
      combined_df["User_ID"].nunique())

print("User_IDs with varying record values:",
      len(inconsistent_users))

print("\nExamples of varying User_IDs:")
display(inconsistent_users.head(10))

# Check whether identical observations have multiple IDs
ids_per_record = (
    combined_df
    .groupby(record_columns, dropna=False)["User_ID"]
    .nunique()
)

print(
    "\nUnique records associated with multiple User_IDs:",
    (ids_per_record > 1).sum()
)

Complete the User_ID consistency check

In [ ]:
# =====================================================
# FINAL USER_ID CONSISTENCY SUMMARY
# =====================================================

print("=" * 60)
print("FINAL USER_ID CONSISTENCY SUMMARY")
print("=" * 60)

print(
    "Total Unique User_IDs:",
    combined_df["User_ID"].nunique()
)

print(
    "User_IDs with Varying Record Values:",
    len(inconsistent_users)
)

print(
    "Unique Records Associated with Multiple User_IDs:",
    (ids_per_record > 1).sum()
)

print("\nExample Records for User_ID 13:")

display(
    combined_df[
        combined_df["User_ID"] == 13
    ]
)

Check how many users have inconsistent records

In [ ]:
print("=" * 60)
print("USER ID CONSISTENCY RESULTS")
print("=" * 60)

print(
    "User_IDs with varying records:",
    len(inconsistent_users)
)

print(
    "Unique feature-and-emotion records linked to multiple User_IDs:",
    int((ids_per_record > 1).sum())
)

Create a new dataset with unique observations

In [ ]:
# =====================================================
# CREATE UNIQUE MODELING DATASET
# =====================================================

from sklearn.model_selection import train_test_split

# Combine cleaned datasets
combined_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# User_ID is excluded from modeling
FEATURE_COLUMNS = [
    "Age",
    "Gender",
    "Platform",
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day"
]

TARGET_COLUMN = "Dominant_Emotion"

MODEL_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]

# Remove identical feature + target observations
model_df = (
    combined_df[MODEL_COLUMNS]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Combined records:", len(combined_df))
print("Unique modeling records:", len(model_df))

print("\nEmotion distribution:")
print(model_df[TARGET_COLUMN].value_counts())

Create stratified training, validation, and testing splits

In [ ]:
# =====================================================
# STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# =====================================================

# First split: 70% training, 30% temporary
train_final, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=42,
    stratify=model_df[TARGET_COLUMN]
)

# Second split: 15% validation, 15% testing
val_final, test_final = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df[TARGET_COLUMN]
)

train_final = train_final.reset_index(drop=True)
val_final = val_final.reset_index(drop=True)
test_final = test_final.reset_index(drop=True)

print("=" * 60)
print("FINAL DATASET SPLIT")
print("=" * 60)

print("Training:", train_final.shape)
print("Validation:", val_final.shape)
print("Testing:", test_final.shape)

print("\nTraining emotions:")
print(train_final[TARGET_COLUMN].value_counts())

print("\nValidation emotions:")
print(val_final[TARGET_COLUMN].value_counts())

print("\nTesting emotions:")
print(test_final[TARGET_COLUMN].value_counts())

Verify that there is no observation-level leakage

In [ ]:
# =====================================================
# VERIFY DATASET OVERLAP
# =====================================================

def get_keys(df):
    return set(
        map(
            tuple,
            df[MODEL_COLUMNS].to_numpy()
        )
    )

train_keys = get_keys(train_final)
val_keys = get_keys(val_final)
test_keys = get_keys(test_final)

print("=" * 60)
print("FINAL DATA LEAKAGE CHECK")
print("=" * 60)

print(
    "Train-Val overlap:",
    len(train_keys & val_keys)
)

print(
    "Train-Test overlap:",
    len(train_keys & test_keys)
)

print(
    "Val-Test overlap:",
    len(val_keys & test_keys)
)

assert len(train_keys & val_keys) == 0
assert len(train_keys & test_keys) == 0
assert len(val_keys & test_keys) == 0

print("\nNo identical feature-and-target records across splits!")

Save the final modeling datasets

In [ ]:
# =====================================================
# SAVE FINAL DATASETS
# =====================================================

from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_final.to_csv(
    OUTPUT_DIR / "train_final.csv",
    index=False
)

val_final.to_csv(
    OUTPUT_DIR / "val_final.csv",
    index=False
)

test_final.to_csv(
    OUTPUT_DIR / "test_final.csv",
    index=False
)

model_df.to_csv(
    OUTPUT_DIR / "unique_modeling_dataset.csv",
    index=False
)

print("Final modeling datasets saved successfully!")

### Exploratory Data Analysis (EDA)

Import libraries for EDA and modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

print("Libraries imported successfully!")

Verify the final datasets

In [ ]:
print("=" * 60)
print("FINAL MODELING DATASET OVERVIEW")
print("=" * 60)

for name, df in {
    "Training": train_final,
    "Validation": val_final,
    "Testing": test_final
}.items():

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

display(train_final.head())

print("\nColumn Data Types:")
print(train_final.dtypes)

Target emotion distribution

In [ ]:
plt.figure(figsize=(10, 5))

order = train_final[TARGET_COLUMN].value_counts().index

ax = sns.countplot(
    data=train_final,
    x=TARGET_COLUMN,
    order=order,
    color="steelblue"
)

for container in ax.containers:
    ax.bar_label(container)

plt.title("Distribution of Dominant Emotions")
plt.xlabel("Dominant Emotion")
plt.ylabel("Number of Users")
plt.xticks(rotation=30)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "emotion_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Gender distribution

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.countplot(
    data=train_final,
    x="Gender",
    order=["Male", "Female", "Non-binary"],
    color="teal"
)

for container in ax.containers:
    ax.bar_label(container)

plt.title("Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

Age distribution

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=train_final,
    x="Age",
    bins=15,
    kde=True
)

plt.title("Age Distribution of Social Media Users")
plt.xlabel("Age")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

Platform usage distribution

In [ ]:
plt.figure(figsize=(10, 5))

platform_order = train_final["Platform"].value_counts().index

sns.countplot(
    data=train_final,
    x="Platform",
    order=platform_order,
    color="mediumpurple"
)

plt.title("Social Media Platform Distribution")
plt.xlabel("Platform")
plt.ylabel("Number of Users")
plt.xticks(rotation=35)

plt.tight_layout()
plt.show()

Daily social media usage by emotion

In [ ]:
plt.figure(figsize=(11, 6))

sns.boxplot(
    data=train_final,
    x=TARGET_COLUMN,
    y="Daily_Usage_Time (minutes)"
)

plt.title("Daily Social Media Usage by Emotion")
plt.xlabel("Dominant Emotion")
plt.ylabel("Daily Usage Time (minutes)")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

Platform versus emotional state

In [ ]:

platform_emotion = pd.crosstab(
    train_df["Platform"],
    train_df[TARGET_COLUMN],
    normalize="index"
) * 100

display(platform_emotion.round(2))

platform_emotion.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    colormap="tab20"
)

plt.title("Emotion Distribution Across Social Media Platforms")
plt.xlabel("Platform")
plt.ylabel("Percentage")
plt.xticks(rotation=30)
plt.legend(
    title="Emotion",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "07_platform_emotion.png",
    dpi=300, bbox_inches="tight"
)
plt.show()


Numerical feature distributions

In [ ]:

fig, axes = plt.subplots(
    2, 3, figsize=(16, 9)
)

for ax, column in zip(axes.flatten(), NUMERIC_COLUMNS):
    sns.histplot(
        train_df[column].dropna(),
        kde=True,
        ax=ax
    )
    ax.set_title(column)

plt.suptitle("Numerical Feature Distributions", fontsize=16)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "08_numeric_distributions.png",
    dpi=300, bbox_inches="tight"
)
plt.show()


Correlation heatmap

In [ ]:
numeric_features = [
    "Age",
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day"
]

plt.figure(figsize=(11, 8))

correlation_matrix = train_final[
    numeric_features
].corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix of Numeric Features")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Feature Preprocessing

Separate features and target variables

In [ ]:
X_train = train_final[FEATURE_COLUMNS].copy()
X_val = val_final[FEATURE_COLUMNS].copy()
X_test = test_final[FEATURE_COLUMNS].copy()

y_train = train_final[TARGET_COLUMN].copy()
y_val = val_final[TARGET_COLUMN].copy()
y_test = test_final[TARGET_COLUMN].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

Encode target labels

In [ ]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

class_names = label_encoder.classes_

print("Emotion Classes:")

for index, emotion in enumerate(class_names):
    print(index, ":", emotion)

joblib.dump(
    label_encoder,
    OUTPUT_DIR / "emotion_label_encoder.pkl"
)

Create preprocessing pipelines

In [ ]:
numeric_features = [
    "Age",
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day"
]

categorical_features = [
    "Gender",
    "Platform"
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Fit preprocessing on training data only

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)

joblib.dump(
    preprocessor,
    OUTPUT_DIR / "preprocessor.pkl"
)

print("Preprocessor saved successfully!")

### Machine Learning Model Development

Define classification models

In [ ]:
models = {
    "Dummy Baseline": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    ),

    "Support Vector Machine": SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_STATE
    ),

    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5
    )
}

print("Total models:", len(models))

Train and evaluate ML models on validation data

In [ ]:
ml_results = []
trained_models = {}

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(
        X_train_processed,
        y_train_encoded
    )

    y_pred = model.predict(
        X_val_processed
    )

    results = {
        "Model": name,
        "Accuracy": accuracy_score(
            y_val_encoded, y_pred
        ),
        "Precision": precision_score(
            y_val_encoded,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_val_encoded,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "F1_Score": f1_score(
            y_val_encoded,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_val_encoded,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

    ml_results.append(results)
    trained_models[name] = model

ml_results_df = pd.DataFrame(ml_results)

ml_results_df = ml_results_df.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

display(ml_results_df)

ml_results_df.to_csv(
    OUTPUT_DIR / "ml_model_comparison.csv",
    index=False
)

Select the best ML model

In [ ]:
best_ml_name = ml_results_df.iloc[0]["Model"]

best_ml_model = trained_models[best_ml_name]

print("Best ML Model:", best_ml_name)

print(
    "Validation Macro F1:",
    ml_results_df.iloc[0]["Macro_F1"]
)

joblib.dump(
    best_ml_model,
    OUTPUT_DIR / "best_ml_model.pkl"
)

Validation confusion matrix

In [ ]:
best_ml_val_pred = best_ml_model.predict(
    X_val_processed
)

cm = confusion_matrix(
    y_val_encoded,
    best_ml_val_pred,
    labels=np.arange(len(class_names))
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    f"Validation Confusion Matrix - {best_ml_name}"
)

plt.xlabel("Predicted Emotion")
plt.ylabel("Actual Emotion")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ml_validation_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Deep Learning Model Development

Import TensorFlow and Keras

In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

tf.keras.utils.set_random_seed(RANDOM_STATE)

print("TensorFlow Version:", tf.__version__)

Build the neural network

In [ ]:
num_features = X_train_processed.shape[1]
num_classes = len(class_names)

dl_model = Sequential([
    Input(shape=(num_features,)),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.30),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        num_classes,
        activation="softmax"
    )
])

dl_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

dl_model.summary()

Configure training callbacks

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath=str(OUTPUT_DIR / "best_dl_model.keras"),
    monitor="val_loss",
    save_best_only=True
)

callbacks = [
    early_stopping,
    checkpoint
]

Train the deep learning model

In [ ]:
history = dl_model.fit(
    X_train_processed,
    y_train_encoded,

    validation_data=(
        X_val_processed,
        y_val_encoded
    ),

    epochs=100,
    batch_size=16,

    callbacks=callbacks,
    verbose=1
)

Plot deep learning training history

In [ ]:
history_df = pd.DataFrame(history.history)

history_df.to_csv(
    OUTPUT_DIR / "dl_training_history.csv",
    index=False
)

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Deep Learning Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "dl_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Plot deep learning loss

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Deep Learning Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "dl_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Evaluate the DL model on validation data

In [ ]:
from tensorflow.keras.models import load_model

best_dl_model = load_model(
    OUTPUT_DIR / "best_dl_model.keras"
)

dl_val_probabilities = best_dl_model.predict(
    X_val_processed
)

dl_val_predictions = np.argmax(
    dl_val_probabilities,
    axis=1
)

dl_val_macro_f1 = f1_score(
    y_val_encoded,
    dl_val_predictions,
    average="macro",
    zero_division=0
)

print(
    "DL Validation Accuracy:",
    accuracy_score(
        y_val_encoded,
        dl_val_predictions
    )
)

print(
    "DL Validation Macro F1:",
    dl_val_macro_f1
)

print(
    classification_report(
        y_val_encoded,
        dl_val_predictions,
        target_names=class_names,
        zero_division=0
    )
)

### Final Model Selection and Testing

Compare the best ML and DL models

In [ ]:
best_ml_macro_f1 = ml_results_df.iloc[0]["Macro_F1"]

comparison = pd.DataFrame({
    "Model": [
        best_ml_name,
        "Deep Learning Neural Network"
    ],

    "Validation_Macro_F1": [
        best_ml_macro_f1,
        dl_val_macro_f1
    ]
})

display(comparison)

comparison.to_csv(
    OUTPUT_DIR / "ml_vs_dl_comparison.csv",
    index=False
)

if dl_val_macro_f1 > best_ml_macro_f1:
    final_model_type = "DL"
    print("Selected Model: Deep Learning")
else:
    final_model_type = "ML"
    print("Selected Model:", best_ml_name)

Evaluate the selected model on the test dataset

In [ ]:
if final_model_type == "DL":

    test_probabilities = best_dl_model.predict(
        X_test_processed
    )

    test_predictions = np.argmax(
        test_probabilities,
        axis=1
    )

else:

    test_predictions = best_ml_model.predict(
        X_test_processed
    )


test_accuracy = accuracy_score(
    y_test_encoded,
    test_predictions
)

test_macro_f1 = f1_score(
    y_test_encoded,
    test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test_encoded,
    test_predictions,
    average="weighted",
    zero_division=0
)

print("=" * 60)
print("FINAL TEST EVALUATION")
print("=" * 60)

print("Selected Model Type:", final_model_type)
print("Test Accuracy:", round(test_accuracy, 4))
print("Test Macro F1:", round(test_macro_f1, 4))
print("Test Weighted F1:", round(test_weighted_f1, 4))

print("\nClassification Report:")

print(
    classification_report(
        y_test_encoded,
        test_predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0
    )
)

Final test confusion matrix

In [ ]:
cm_test = confusion_matrix(
    y_test_encoded,
    test_predictions,
    labels=np.arange(len(class_names))
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Final Model - Test Confusion Matrix")
plt.xlabel("Predicted Emotion")
plt.ylabel("Actual Emotion")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Save final evaluation results

In [ ]:
final_results = pd.DataFrame([{
    "Selected_Model": (
        "Deep Learning Neural Network"
        if final_model_type == "DL"
        else best_ml_name
    ),
    "Test_Samples": len(y_test_encoded),
    "Test_Accuracy": test_accuracy,
    "Test_Macro_F1": test_macro_f1,
    "Test_Weighted_F1": test_weighted_f1
}])

display(final_results)

final_results.to_csv(
    OUTPUT_DIR / "final_model_evaluation.csv",
    index=False
)

print("Final evaluation results saved successfully!")

Test prediction on a new user

In [ ]:
new_user = pd.DataFrame([{
    "Age": 25,
    "Gender": "Male",
    "Platform": "Instagram",
    "Daily_Usage_Time (minutes)": 120,
    "Posts_Per_Day": 3,
    "Likes_Received_Per_Day": 50,
    "Comments_Received_Per_Day": 10,
    "Messages_Sent_Per_Day": 20
}])

new_user_processed = preprocessor.transform(
    new_user[FEATURE_COLUMNS]
)

if final_model_type == "DL":

    probabilities = best_dl_model.predict(
        new_user_processed,
        verbose=0
    )

    predicted_class = int(
        np.argmax(probabilities, axis=1)[0]
    )

else:

    predicted_class = int(
        best_ml_model.predict(new_user_processed)[0]
    )

predicted_emotion = label_encoder.inverse_transform(
    [predicted_class]
)[0]

print("Predicted Dominant Emotion:", predicted_emotion)

### Final Model Validation and Deployment Preparation

In [ ]:
# =====================================================
# PREDICTION CONFIDENCE ANALYSIS
# =====================================================

if final_model_type == "DL":

    probabilities = best_dl_model.predict(
        new_user_processed,
        verbose=0
    )[0]

else:

    if hasattr(best_ml_model, "predict_proba"):
        probabilities = best_ml_model.predict_proba(
            new_user_processed
        )[0]
    else:
        probabilities = None

if probabilities is not None:

    confidence_df = pd.DataFrame({
        "Emotion": class_names,
        "Probability": probabilities
    })

    confidence_df = confidence_df.sort_values(
        "Probability",
        ascending=False
    )

    display(confidence_df)

    plt.figure(figsize=(10, 5))

    sns.barplot(
        data=confidence_df,
        x="Emotion",
        y="Probability",
        color="steelblue"
    )

    plt.title("Predicted Emotion Probabilities")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

else:
    print("Probability estimates are unavailable.")

Generate feature importance using permutation importance

In [ ]:
# =====================================================
# PERMUTATION FEATURE IMPORTANCE
# =====================================================

from sklearn.inspection import permutation_importance

if final_model_type == "ML":

    importance = permutation_importance(
        best_ml_model,
        X_val_processed,
        y_val_encoded,
        n_repeats=10,
        random_state=42,
        scoring="f1_macro"
    )

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance.importances_mean
    })

    importance_df = importance_df.sort_values(
        "Importance",
        ascending=False
    )

    display(importance_df)

    plt.figure(figsize=(10, 7))

    sns.barplot(
        data=importance_df.head(15),
        x="Importance",
        y="Feature",
        color="teal"
    )

    plt.title("Top 15 Permutation Feature Importances")
    plt.tight_layout()
    plt.show()

else:
    print(
        "Selected model is Deep Learning. "
        "This cell evaluates ML feature importance only."
    )

Save a complete prediction pipeline

In [ ]:
# =====================================================
# SAVE MODEL METADATA
# =====================================================

import json
import joblib

model_metadata = {
    "project": "Social Media Usage and Emotional Well-Being",
    "model_type": final_model_type,
    "selected_model": (
        "Deep Learning Neural Network"
        if final_model_type == "DL"
        else best_ml_name
    ),
    "features": FEATURE_COLUMNS,
    "target": TARGET_COLUMN,
    "classes": class_names.tolist(),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1)
}

with open(
    OUTPUT_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(model_metadata, file, indent=4)

joblib.dump(
    preprocessor,
    OUTPUT_DIR / "preprocessor.pkl"
)

joblib.dump(
    label_encoder,
    OUTPUT_DIR / "emotion_label_encoder.pkl"
)

if final_model_type == "ML":

    joblib.dump(
        best_ml_model,
        OUTPUT_DIR / "final_ml_model.pkl"
    )

else:

    best_dl_model.save(
        OUTPUT_DIR / "final_dl_model.keras"
    )

print("Final model artifacts saved successfully!")

Verify that the saved model can be loaded

In [ ]:
# =====================================================
# VERIFY SAVED MODEL
# =====================================================

loaded_preprocessor = joblib.load(
    OUTPUT_DIR / "preprocessor.pkl"
)

loaded_encoder = joblib.load(
    OUTPUT_DIR / "emotion_label_encoder.pkl"
)

processed_sample = loaded_preprocessor.transform(
    new_user[FEATURE_COLUMNS]
)

if final_model_type == "ML":

    loaded_model = joblib.load(
        OUTPUT_DIR / "final_ml_model.pkl"
    )

    prediction = loaded_model.predict(
        processed_sample
    )[0]

else:

    from tensorflow.keras.models import load_model

    loaded_model = load_model(
        OUTPUT_DIR / "final_dl_model.keras"
    )

    prediction = np.argmax(
        loaded_model.predict(
            processed_sample,
            verbose=0
        ),
        axis=1
    )[0]

emotion = loaded_encoder.inverse_transform(
    [int(prediction)]
)[0]

print("Loaded Model Prediction:", emotion)
print("Original Model Prediction:", predicted_emotion)

assert emotion == predicted_emotion

print("Model loading verification successful!")

Generate a final project summary

In [ ]:
# =====================================================
# FINAL PROJECT SUMMARY
# =====================================================

project_summary = pd.DataFrame([{
    "Project": "Social Media Usage and Emotional Well-Being",
    "Task": "Multiclass Classification",
    "Unique_Records": len(model_df),
    "Training_Records": len(train_final),
    "Validation_Records": len(val_final),
    "Testing_Records": len(test_final),
    "Number_of_Features": len(FEATURE_COLUMNS),
    "Number_of_Classes": len(class_names),
    "Selected_Model": (
        best_ml_name
        if final_model_type == "ML"
        else "Deep Learning Neural Network"
    ),
    "Test_Accuracy": round(test_accuracy, 4),
    "Test_Macro_F1": round(test_macro_f1, 4),
    "Test_Weighted_F1": round(test_weighted_f1, 4)
}])

display(project_summary)

project_summary.to_csv(
    OUTPUT_DIR / "project_summary.csv",
    index=False
)

print("Project summary saved successfully!")